In [0]:
%sql
CREATE OR REPLACE TABLE workspace.tp_futbol.gold_partidos AS
WITH base AS (
  SELECT
    p.*,
    r.regimen    AS regimen_publico,
    r.visitantes AS visitantes_permitidos,
    r.confianza  AS confianza_regimen,
    el.latitud AS lat_local,     el.longitud AS lon_local,
    ev.latitud AS lat_visitante, ev.longitud AS lon_visitante,
    CASE WHEN p.psc_local  IS NOT NULL THEN 'pinnacle_cierre'
         WHEN p.avgc_local IS NOT NULL THEN 'promedio_cierre' END AS fuente_cuota,
    CASE WHEN p.psc_local IS NOT NULL THEN p.psc_local     ELSE p.avgc_local     END AS cuota_local,
    CASE WHEN p.psc_local IS NOT NULL THEN p.psc_empate    ELSE p.avgc_empate    END AS cuota_empate,
    CASE WHEN p.psc_local IS NOT NULL THEN p.psc_visitante ELSE p.avgc_visitante END AS cuota_visitante
  FROM workspace.tp_futbol.silver_partidos p
  LEFT JOIN workspace.tp_futbol.silver_restricciones r
         ON p.liga = r.liga AND p.fecha BETWEEN r.desde AND r.hasta
  LEFT JOIN workspace.tp_futbol.silver_equipos el ON el.liga = p.liga AND el.equipo = p.local
  LEFT JOIN workspace.tp_futbol.silver_equipos ev ON ev.liga = p.liga AND ev.equipo = p.visitante
)
SELECT
  partido_id, liga, torneo, temporada, fecha, year(fecha) AS anio,
  local, visitante, goles_local, goles_visitante, resultado,
  goles_local - goles_visitante                          AS diferencia_goles,
  CASE resultado WHEN 'H' THEN 3 WHEN 'D' THEN 1 ELSE 0 END AS puntos_local,
  regimen_publico, visitantes_permitidos, confianza_regimen,
  round(2 * 6371 * asin(sqrt(
      pow(sin(radians(lat_visitante - lat_local) / 2), 2)
    + cos(radians(lat_local)) * cos(radians(lat_visitante))
    * pow(sin(radians(lon_visitante - lon_local) / 2), 2))), 1) AS distancia_km,
  fuente_cuota, cuota_local, cuota_empate, cuota_visitante,
  1/cuota_local + 1/cuota_empate + 1/cuota_visitante - 1                        AS margen_casa,
  (1/cuota_local)     / (1/cuota_local + 1/cuota_empate + 1/cuota_visitante)    AS prob_local,
  (1/cuota_empate)    / (1/cuota_local + 1/cuota_empate + 1/cuota_visitante)    AS prob_empate,
  (1/cuota_visitante) / (1/cuota_local + 1/cuota_empate + 1/cuota_visitante)    AS prob_visitante,
  1/psc_local   + 1/psc_empate   + 1/psc_visitante   - 1  AS margen_pinnacle,
  1/avgc_local  + 1/avgc_empate  + 1/avgc_visitante  - 1  AS margen_promedio_mercado,
  1/b365c_local + 1/b365c_empate + 1/b365c_visitante - 1  AS margen_b365,
  b365c_local, b365c_empate, b365c_visitante
FROM base;

In [0]:
%sql
SELECT
  count(*)                                          AS partidos,
  count_if(regimen_publico IS NULL)                 AS sin_regimen,
  count_if(distancia_km IS NULL)                    AS sin_distancia,
  count_if(fuente_cuota IS NULL)                    AS sin_cuotas,
  count_if(fuente_cuota = 'pinnacle_cierre')        AS con_pinnacle,
  count_if(fuente_cuota = 'promedio_cierre')        AS con_promedio,
  round(max(distancia_km))                          AS distancia_max_km
FROM workspace.tp_futbol.gold_partidos;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.tp_futbol.gold_resumen_liga_temporada AS
SELECT
  liga, temporada,
  count(*)                                         AS partidos,
  round(avg(CASE WHEN resultado = 'H' THEN 1 ELSE 0 END), 4) AS pct_local,
  round(avg(CASE WHEN resultado = 'D' THEN 1 ELSE 0 END), 4) AS pct_empate,
  round(avg(CASE WHEN resultado = 'A' THEN 1 ELSE 0 END), 4) AS pct_visitante,
  round(avg(goles_local + goles_visitante), 2)     AS goles_promedio,
  round(avg(diferencia_goles), 3)                  AS diferencia_goles_promedio,
  round(avg(prob_local), 4)                        AS prob_local_mercado,
  round(avg(margen_casa), 4)                       AS margen_promedio,
  round(avg(distancia_km), 1)                      AS distancia_promedio_km
FROM workspace.tp_futbol.gold_partidos
GROUP BY liga, temporada;

CREATE OR REPLACE TABLE workspace.tp_futbol.gold_resumen_regimen_publico AS
SELECT
  liga, regimen_publico, visitantes_permitidos,
  count(*)                                         AS partidos,
  min(fecha)                                       AS desde,
  max(fecha)                                       AS hasta,
  round(avg(CASE WHEN resultado = 'H' THEN 1 ELSE 0 END), 4) AS pct_local,
  round(avg(CASE WHEN resultado = 'D' THEN 1 ELSE 0 END), 4) AS pct_empate,
  round(avg(CASE WHEN resultado = 'A' THEN 1 ELSE 0 END), 4) AS pct_visitante,
  round(avg(goles_local + goles_visitante), 2)     AS goles_promedio,
  round(avg(diferencia_goles), 3)                  AS diferencia_goles_promedio,
  round(avg(prob_local), 4)                        AS prob_local_mercado,
  round(avg(margen_casa), 4)                       AS margen_promedio
FROM workspace.tp_futbol.gold_partidos
GROUP BY liga, regimen_publico, visitantes_permitidos;

In [0]:
%sql
SELECT liga, regimen_publico, visitantes_permitidos, partidos, pct_local, prob_local_mercado, diferencia_goles_promedio
FROM workspace.tp_futbol.gold_resumen_regimen_publico
ORDER BY liga, regimen_publico, visitantes_permitidos;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.tp_futbol.gold_mercado_trimestral AS
SELECT
  liga,
  regimen_publico,
  CAST(date_trunc('QUARTER', fecha) AS DATE)                      AS trimestre,
  count(*)                                                        AS partidos,
  round(avg(CASE WHEN resultado = 'H' THEN 1 ELSE 0 END), 4)      AS pct_local,
  round(avg(prob_local), 4)                                       AS prob_local_mercado,
  round((avg(prob_local) - avg(CASE WHEN resultado = 'H' THEN 1 ELSE 0 END)) * 100, 2) AS error_mercado_local_pp,
  round(avg(margen_pinnacle), 4)                                  AS margen_pinnacle,
  round(avg(margen_promedio_mercado), 4)                          AS margen_promedio_mercado,
  round(avg(margen_b365), 4)                                      AS margen_b365
FROM workspace.tp_futbol.gold_partidos
WHERE prob_local IS NOT NULL
GROUP BY liga, regimen_publico, CAST(date_trunc('QUARTER', fecha) AS DATE);

In [0]:
%sql
SELECT trimestre,
       sum(partidos) AS partidos,
       round(sum(error_mercado_local_pp * partidos) / sum(partidos), 2) AS error_mercado_pp
FROM workspace.tp_futbol.gold_mercado_trimestral
WHERE regimen_publico = 'sin_publico'
GROUP BY trimestre
ORDER BY trimestre;